[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prashantkul/ai-ml-interviews-study-guide/blob/main/notebooks/week4_multiple_testing.ipynb)


# Week 4 - Multiple Testing Correction: Bonferroni vs Benjamini-Hochberg FDR

**Scenario.** You built a monitor (call it LAD) and you want to know whether it beats a baseline. You did not test on one task; you tested on **K = 12 attack categories**. For each category you ran a hypothesis test of the form *'is LAD better than baseline on this attack?'* and got 12 p-values.

Naively you set alpha = 0.05 and call anything below 0.05 a 'win'. But if all 12 nulls were actually true, you'd still expect roughly 0.05 * 12 = 0.6 false positives just by chance. With more comparisons the chance of *at least one* false positive (the **Family-Wise Error Rate**, FWER) blows up to roughly 1 - (1 - 0.05)^12 ~ 0.46.

Two corrections fix this in different ways:

- **Bonferroni** controls FWER strictly: reject H_i only if p_i <= alpha / K. Simple, conservative, kills power.
- **Benjamini-Hochberg (BH)** controls the **False Discovery Rate** (FDR), i.e. the *expected proportion* of rejected hypotheses that are actually null. More powerful when K is large and several effects are real.

This notebook implements both from scratch, applies them to a synthetic 'LAD across 12 attacks' setup, then runs a Monte Carlo to compare FWER, FDR, and power.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(21)

## 1. Synthetic setup: 12 attack categories

Ground truth: 4 of the 12 attack categories have a real effect (LAD genuinely beats baseline), 8 are null.

For each category we draw a one-sided z-statistic:
- **Null categories**: z ~ N(0, 1)
- **Alternative categories**: z ~ N(2.5, 1)

Then convert to a one-sided p-value via `1 - Phi(z) = norm.sf(z)`.

(You can think of each z as the standardized log-odds-ratio from a two-proportion test on n_per_category=200 trials per category; we are skipping the bookkeeping and going straight to the test statistic.)

In [ ]:
K = 12
n_per_category = 200  # conceptual sample size per attack category
n_alt = 4
n_null = K - n_alt
alt_mean = 2.5

truth = np.array(['alt'] * n_alt + ['null'] * n_null)
is_alt = truth == 'alt'

z = np.empty(K)
z[is_alt] = np.random.normal(loc=alt_mean, scale=1.0, size=n_alt)
z[~is_alt] = np.random.normal(loc=0.0, scale=1.0, size=n_null)

p = stats.norm.sf(z)  # one-sided p-values

categories = [f'attack_{i:02d}' for i in range(K)]

print(f"{'category':<12} {'truth':<6} {'z':>8} {'p':>10}")
print('-' * 40)
for c, t, zi, pi in zip(categories, truth, z, p):
    print(f'{c:<12} {t:<6} {zi:>8.3f} {pi:>10.4f}')

## 2. Naive threshold (no correction)

Reject H_i whenever p_i < 0.05. Count how many of those rejections are real vs spurious.

In [ ]:
alpha = 0.05
naive_reject = p < alpha

naive_total = int(naive_reject.sum())
naive_tp = int((naive_reject & is_alt).sum())
naive_fp = int((naive_reject & ~is_alt).sum())

print(f'Naive (alpha=0.05) rejected: {naive_total}')
print(f'  True positives:  {naive_tp} / {n_alt}')
print(f'  False positives: {naive_fp} / {n_null}')
if naive_total:
    print(f'  False discovery proportion: {naive_fp / naive_total:.3f}')

## 3. Bonferroni correction

**Rule.** Reject H_i iff
$$ p_i \;\le\; \frac{\alpha}{K}. $$

**Why it works.** By the union bound, if all K nulls hold,
$$ P(\text{any rejection}) \;\le\; \sum_{i=1}^K P(p_i \le \alpha/K) \;=\; K \cdot \frac{\alpha}{K} \;=\; \alpha. $$
So FWER <= alpha. The price is severe: you must hit a 0.05 / 12 ~ 0.00417 bar on every category.

In [ ]:
bonf_thresh = alpha / K
bonf_reject = p <= bonf_thresh

bonf_total = int(bonf_reject.sum())
bonf_tp = int((bonf_reject & is_alt).sum())
bonf_fp = int((bonf_reject & ~is_alt).sum())

print(f'Bonferroni threshold = alpha/K = {bonf_thresh:.5f}')
print(f'Bonferroni rejected: {bonf_total}')
print(f'  True positives:  {bonf_tp} / {n_alt}')
print(f'  False positives: {bonf_fp} / {n_null}')
if bonf_total:
    print(f'  False discovery proportion: {bonf_fp / bonf_total:.3f}')

## 4. Benjamini-Hochberg FDR (from scratch)

**Procedure.** Sort the p-values ascending: $p_{(1)} \le p_{(2)} \le \dots \le p_{(K)}$. Pick FDR level $q$ (here $q = 0.05$). Find the **largest** index $i^*$ such that
$$ p_{(i)} \;\le\; \frac{i}{K}\, q. $$
Reject $H_{(1)}, H_{(2)}, \dots, H_{(i^*)}$. If no such $i$ exists, reject nothing.

**Guarantee.** Under independence (or positive dependence) of the test statistics, BH controls
$$ \text{FDR} \;=\; \mathbb{E}\!\left[\frac{V}{\max(R, 1)}\right] \;\le\; \frac{K_0}{K}\, q \;\le\; q, $$
where $V$ is the number of false rejections, $R$ the total number of rejections, and $K_0$ the (unknown) number of true nulls.

In [ ]:
def bh_fdr(pvals, q=0.05):
    """Benjamini-Hochberg: returns boolean rejection mask aligned with the input order."""
    pvals = np.asarray(pvals)
    K_local = len(pvals)
    order = np.argsort(pvals)
    sorted_p = pvals[order]
    thresholds = (np.arange(1, K_local + 1) / K_local) * q
    below = sorted_p <= thresholds
    if not below.any():
        return np.zeros(K_local, dtype=bool), -1, thresholds, order
    i_star = int(np.max(np.where(below)[0]))  # 0-indexed largest passing rank
    reject = np.zeros(K_local, dtype=bool)
    reject[order[: i_star + 1]] = True
    return reject, i_star, thresholds, order


q = 0.05
bh_reject, i_star, bh_thresholds, order = bh_fdr(p, q=q)

print('Sorted p-values vs BH thresholds (i/K)*q:')
print(f"{'rank':>4} {'category':<12} {'p_(i)':>10} {'(i/K)*q':>10} {'pass?':>6}")
for rank, idx in enumerate(order, start=1):
    flag = 'yes' if p[idx] <= (rank / K) * q else 'no'
    print(f'{rank:>4} {categories[idx]:<12} {p[idx]:>10.4f} {(rank / K) * q:>10.4f} {flag:>6}')

print()
print(f'Largest passing rank i* = {i_star + 1 if i_star >= 0 else 0}')

bh_total = int(bh_reject.sum())
bh_tp = int((bh_reject & is_alt).sum())
bh_fp = int((bh_reject & ~is_alt).sum())
print(f'BH rejected: {bh_total}')
print(f'  True positives:  {bh_tp} / {n_alt}')
print(f'  False positives: {bh_fp} / {n_null}')
if bh_total:
    print(f'  False discovery proportion: {bh_fp / bh_total:.3f}')

## 5. Visualization: sorted p-values, BH line, Bonferroni line

In [ ]:
ranks = np.arange(1, K + 1)
sorted_p = p[order]
sorted_truth = is_alt[order]
sorted_bh_reject = bh_reject[order]
sorted_bonf_reject = bonf_reject[order]

fig, ax = plt.subplots(figsize=(8, 5))

bh_label_used = False
bonf_label_used = False
for rk, pv, truth_alt, bh_rej, bf_rej in zip(
    ranks, sorted_p, sorted_truth, sorted_bh_reject, sorted_bonf_reject
):
    color = 'tab:blue' if truth_alt else 'tab:gray'
    marker = 'o' if truth_alt else 's'
    ax.scatter(rk, pv, color=color, marker=marker, s=80, zorder=3,
               edgecolor='black', linewidth=0.6)
    if bh_rej:
        ax.scatter(rk, pv, facecolors='none', edgecolors='tab:green', s=240,
                   linewidth=1.8, zorder=4,
                   label=None if bh_label_used else 'BH reject')
        bh_label_used = True
    if bf_rej:
        ax.scatter(rk, pv, facecolors='none', edgecolors='tab:red', s=380,
                   linewidth=1.8, zorder=5,
                   label=None if bonf_label_used else 'Bonferroni reject')
        bonf_label_used = True

ax.plot(ranks, (ranks / K) * q, color='tab:green', linestyle='--',
        label=f'BH line: (i/K)*q, q={q}')
ax.axhline(bonf_thresh, color='tab:red', linestyle=':',
           label=f'Bonferroni: alpha/K = {bonf_thresh:.4f}')
ax.axhline(alpha, color='tab:gray', linestyle=':', alpha=0.6,
           label=f'Naive alpha = {alpha}')

ax.set_xlabel('rank i (sorted ascending by p-value)')
ax.set_ylabel('p-value')
ax.set_title('BH vs Bonferroni on K=12 attack categories\nblue circles = true alt, gray squares = true null')
ax.set_xticks(ranks)
ax.legend(loc='upper left', fontsize=8)
ax.set_ylim(-0.02, max(1.0, sorted_p.max() * 1.1))
plt.tight_layout()
plt.show()

## 6. Monte Carlo: FWER, FDR, and power across 2,000 replications

Re-run the entire 12-category experiment 2,000 times with fresh draws and track:
- **FWER** = P(at least one false rejection) for each method,
- **FDR** = E[V / max(R, 1)] for each method,
- **Power** = E[fraction of the 4 true alternatives correctly rejected] for each method.

In [ ]:
n_sim = 2000
rng = np.random.default_rng(21)

methods = ['uncorrected', 'bonferroni', 'bh']
fwer_count = {m: 0 for m in methods}
fdp_sum = {m: 0.0 for m in methods}
power_sum = {m: 0.0 for m in methods}

for _ in range(n_sim):
    z_sim = np.empty(K)
    z_sim[is_alt] = rng.normal(loc=alt_mean, scale=1.0, size=n_alt)
    z_sim[~is_alt] = rng.normal(loc=0.0, scale=1.0, size=n_null)
    p_sim = stats.norm.sf(z_sim)

    rej = {
        'uncorrected': p_sim < alpha,
        'bonferroni': p_sim <= alpha / K,
        'bh': bh_fdr(p_sim, q=alpha)[0],
    }

    for m in methods:
        r = rej[m]
        V = int((r & ~is_alt).sum())
        R = int(r.sum())
        TP = int((r & is_alt).sum())
        if V > 0:
            fwer_count[m] += 1
        fdp_sum[m] += V / max(R, 1)
        power_sum[m] += TP / n_alt

fwer = {m: fwer_count[m] / n_sim for m in methods}
fdr = {m: fdp_sum[m] / n_sim for m in methods}
power = {m: power_sum[m] / n_sim for m in methods}

print(f"{'method':<12} {'FWER':>8} {'FDR':>8} {'power':>8}")
print('-' * 38)
for m in methods:
    print(f'{m:<12} {fwer[m]:>8.3f} {fdr[m]:>8.3f} {power[m]:>8.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

x = np.arange(len(methods))
width = 0.25

fwer_vals = [fwer[m] for m in methods]
fdr_vals = [fdr[m] for m in methods]
power_vals = [power[m] for m in methods]

ax.bar(x - width, fwer_vals, width, label='FWER', color='tab:red')
ax.bar(x, fdr_vals, width, label='FDR', color='tab:orange')
ax.bar(x + width, power_vals, width, label='Power', color='tab:green')

ax.axhline(alpha, color='black', linestyle='--', linewidth=0.8,
           label=f'alpha = q = {alpha}')
ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.set_ylabel('rate (over 2000 replications)')
ax.set_title('FWER vs FDR vs Power across correction methods')
ax.legend()
ax.set_ylim(0, 1.05)
for i, m in enumerate(methods):
    ax.text(i - width, fwer[m] + 0.02, f'{fwer[m]:.2f}', ha='center', fontsize=8)
    ax.text(i, fdr[m] + 0.02, f'{fdr[m]:.2f}', ha='center', fontsize=8)
    ax.text(i + width, power[m] + 0.02, f'{power[m]:.2f}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

**Reading the bar chart.**

- *Uncorrected:* FWER is large (well above 0.05) because with 8 true nulls there are many chances to spuriously reject. Power is high but the false-alarm rate is unacceptable.
- *Bonferroni:* FWER is squashed below 0.05 (strict family-wise control). Power drops noticeably because the per-test bar is alpha/12 ~ 0.004.
- *BH-FDR:* FDR sits at or below q = 0.05 in expectation, FWER sits between Bonferroni and uncorrected, and power is meaningfully **higher** than Bonferroni. BH spends its error budget on the *proportion* of rejections that are wrong, not on the *probability of any* error - and that's the more useful guarantee when you have many tests and several real effects.

## 7. Dependence warning

The BH guarantee `FDR <= (K_0/K) * q` is proven under:
1. **Independence** of the test statistics across hypotheses, or
2. **Positive Regression Dependence on the Subset (PRDS)** of nulls - a natural form of positive dependence (e.g. one-sided tests on positively correlated Gaussians).

If your tests have arbitrary or negative dependence, vanilla BH can over-reject. The fix is **Benjamini-Yekutieli (2001)**, which inflates the BH thresholds by `c(K) = sum_{i=1}^K 1/i`:
$$ p_{(i)} \;\le\; \frac{i}{K \cdot c(K)}\, q. $$
BY is conservative under any dependence structure. Use BY when, e.g., your 12 attack categories share examples, share a model, or share noise. Use BH when the categories are evaluated on disjoint samples and the test statistics are plausibly independent.

## 8. Flashcard summary

> **Q.** When I have K comparisons, I control FDR using BH because Bonferroni is too conservative when...
>
> **A.** ...K is large *and* a non-trivial fraction of the alternatives are real. Bonferroni controls the *probability of any* false positive at alpha by demanding p_i <= alpha/K on every test, which destroys power as K grows. BH instead controls the *expected proportion* of rejections that are false, so it can let in more discoveries while still bounding the false-discovery rate at q. BH = better recall on a multi-test screen; Bonferroni = bullet-proof single-FP guarantee for safety-critical or confirmatory testing.

## 9. Interview rehearsal

> **Interviewer.** 'You tested LAD on 12 attack types and 5 came back significant at p < 0.05. Are they real?'
>
> **Answer.** Not necessarily. With 12 simultaneous tests at alpha = 0.05, even if none of the categories had a real effect I'd expect roughly 0.6 false positives by chance, and the probability of getting *at least one* spurious 'win' is about 1 - 0.95^12 ~ 0.46. So a raw count of 5 'significant' attacks tells me almost nothing on its own.
>
> What I do is correct for multiplicity. **Bonferroni** would demand p_i <= 0.05/12 ~ 0.0042 on every category - that controls the family-wise error rate at 5% but kills power. **Benjamini-Hochberg at q = 0.05** is what I'd actually report: I sort the 12 p-values, find the largest i with p_(i) <= (i/12) * 0.05, and reject all hypotheses up to that rank. BH controls the *false discovery rate* in expectation, so among the survivors I expect at most ~5% to be spurious.
>
> The cell below prints the exact numbers from this notebook: how many categories the naive cut flagged, how many Bonferroni flagged, how many BH flagged, and the BH/Bonferroni power across the 2,000 replications. Same FDR / FWER target, but BH recovers meaningfully more of the real wins.
>
> One caveat: BH assumes independence or positive dependence across the 12 tests. If the attack categories share examples or a shared model surface, I'd switch to **Benjamini-Yekutieli**, which adds a `sum_{i=1}^K 1/i` factor to the thresholds and is valid under arbitrary dependence.

In [ ]:
print(
    f"On this seed: naive={naive_total}, bonferroni={bonf_total}, bh={bh_total} "
    f"(BH true positives={bh_tp}, false positives={bh_fp}).\n"
    f"Across {n_sim} replications: "
    f"power[bonferroni]={power['bonferroni']:.3f}, power[bh]={power['bh']:.3f}, "
    f"power[uncorrected]={power['uncorrected']:.3f}.\n"
    f"FWER[uncorrected]={fwer['uncorrected']:.3f}, FWER[bonferroni]={fwer['bonferroni']:.3f}, "
    f"FWER[bh]={fwer['bh']:.3f}.\n"
    f"FDR[uncorrected]={fdr['uncorrected']:.3f}, FDR[bonferroni]={fdr['bonferroni']:.3f}, "
    f"FDR[bh]={fdr['bh']:.3f}."
)